# RSNA knee: efficiency pipeline

One streaming pass per GPU (`efficiency/pipeline.py`, code dataset `rsna-knee-code-eff`):

1. **Series**: one sagittal + one coronal per study (fluid-sensitive first); no axial series.
2. **Read** (2 reader processes per GPU, 16 threads each): one `read()` per DICOM, headers parsed directly and pixels taken from the PixelData offset (bit-identical to pydicom); slices past the 64 kept: headers only. Reading starts before the models load.
3. **GPU**: percentile normalisation, 384 px cache, 32 x 128 x 128 volume -> side -> flip to a right knee -> ROI U-Net -> boxes (region bounds on the GPU, only the box sub-volumes copied back) -> per-knee consensus / registration -> crops -> one crop model per finding, weights in float16 (no autocast); the joint model's encoder runs once per crop for its four findings.
4. **Recipe** (`recipe.json`): ACL `mr85`, MCL `pl86`, menisci `cafax87`, OA x 3 `mk90`, Baker's `91`; effusion / synovitis / contusion / fracture from `jc1` (joint capsule crop, Codex labels).

No pip install (the ROI U-Net architecture is vendored in the code dataset); the JPEG decoders are installed from the wheels dataset only if a compressed DICOM shows up.

In [ ]:
import os
import subprocess
import sys
import time

t0 = time.time()
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    CODE = "/kaggle/input/datasets/theoviel/rsna-knee-code-eff/"
    WEIGHTS = "/kaggle/input/datasets/theoviel/rsna-knee-weights-eff/"
    DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/"
    OUT_DIR = "/tmp/knee/"
    os.environ["KNEE_WHEELS"] = "/kaggle/input/datasets/theoviel/rsna-knee-wheels"
else:
    CODE = "kaggle/code/"
    WEIGHTS = "kaggle/weights/"
    DATA_DIR = "../input/"
    OUT_DIR = "../output/inference_eff/"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"       # the code dataset is read-only
RECIPE = "recipe.json"     # or "recipe_plus.json": + mk87 (menisci) and j28 (joint findings), ~6.7 H100 ms / study more

In [ ]:
# GPU count without importing torch in the notebook process (~2 s)
if os.environ.get("CUDA_VISIBLE_DEVICES"):
    GPUS = os.environ["CUDA_VISIBLE_DEVICES"]
else:
    n_gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")
    GPUS = ",".join(map(str, range(max(n_gpu, 1))))
cmd = [sys.executable, os.path.join(CODE, "efficiency", "pipeline.py"), "--data_dir", DATA_DIR,
       "--series_dir", "test_series", "--series_csv", os.path.join(DATA_DIR, "test_series.csv"),
       "--weights", WEIGHTS, "--recipe", os.path.join(WEIGHTS, RECIPE), "--out_dir", OUT_DIR, "--submission", "submission.csv", "--gpus", GPUS]
subprocess.run(cmd, check=True)
print(f"{time.time() - t0:.0f}s")

In [ ]:
import pandas as pd

sub = pd.read_csv("submission.csv")
print(sub.shape, sub.iloc[:, 1:].mean().round(3).to_dict())